# Previsão de Vendas com Machine Learning

**Python · scikit-learn · Regressão supervisionada**

**Objetivo:** prever `Sales` a partir dos investimentos em `TV`, `Radio` e `Newspaper`. O estudo compara modelos simples, regressões e rede neural, com validação cruzada e teste reservado. Trata-se de **previsão**, não de estimativa causal do efeito da publicidade.

## Preparação

**Dependências:** NumPy, pandas, Plotly, Pingouin, statsmodels e scikit-learn. Se necessário, execute `%pip install numpy pandas plotly pingouin statsmodels scikit-learn` no ambiente Jupyter.

**Dados:** o arquivo `advertising.csv` deve estar no mesmo diretório do notebook. Para publicação reproduzível, documente a origem e as condições de uso da base.

In [1]:
# Bibliotecas para trabalhar com os dados
import numpy as np
import pandas as pd
import pingouin as pg

# Seleção e avaliação dos modelos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

# Preparação dos dados
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Modelos preditivos
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gráficos
import plotly.express as px
import plotly.io as pio

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

pio.templates.default = 'plotly_dark'
px.defaults.height = 500
px.defaults.color_continuous_scale = px.colors.sequential.Viridis

## Dados e divisão treino/teste

A base contém investimentos em três canais e vendas (`Sales`). Como o arquivo não documenta moeda, unidade, período ou amostragem, os resultados permanecem **nas unidades originais**, sem interpretação monetária.

In [2]:
df = pd.read_csv('advertising.csv')

df.info()

display(df.head())
display(df.isna().sum())
display(df.nunique())

print(f'Linhas e colunas: {df.shape}')
print(f'Linhas duplicadas: {df.duplicated().sum()}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TV         200 non-null    float64
 1   Radio      200 non-null    float64
 2   Newspaper  200 non-null    float64
 3   Sales      200 non-null    float64
dtypes: float64(4)
memory usage: 6.4 KB


,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

TV           190
Radio        167
Newspaper    172
Sales        121
dtype: int64

Linhas e colunas: (200, 4)
Linhas duplicadas: 0


Na execução registrada, são **200 observações**, sem ausências ou linhas duplicadas. Reservamos **40 para teste (20%)** e usamos **160 para exploração, treino e validação**.

A separação aleatória pressupõe observações independentes. Como o conjunto de dados já era conhecido antes deste estudo, o teste reservado **não equivale a validação externa independente**.

In [3]:
X = df.drop(columns=['Sales'])
Y = df['Sales']

X_train, X_test, Y_train, Y_test = train_test_split(
      X
    , Y
    , test_size=0.2
    , random_state=42
)

# Essa será a base da análise exploratória. O teste fica guardado.
df_train = X_train.copy()
df_train['Sales'] = Y_train

print(f'Linhas para treino e análise: {len(df_train)}')
print(f'Linhas reservadas para teste: {len(X_test)}')

Linhas para treino e análise: 160
Linhas reservadas para teste: 40


## 1. Análise descritiva

Média, mediana, dispersão, quartis e assimetria das variáveis, calculadas apenas no conjunto de treino.

In [4]:
display(df_train.describe().round(2))

# Assimetria positiva indica uma cauda maior à direita da distribuição.
display(df_train.skew().round(2))

,TV,Radio,Newspaper,Sales
count,160.00,160.00,160.00,160.00
mean,150.02,22.88,29.95,15.33
std,84.42,14.81,20.34,5.19
min,0.70,0.00,0.30,1.60
25%,77.75,9.82,12.88,11.58
50%,150.65,21.20,25.60,16.20
75%,218.82,36.42,44.50,19.05
max,296.40,49.60,100.90,27.00


TV          -0.09
Radio        0.13
Newspaper    0.71
Sales       -0.13
dtype: float64

### Valores extremos

Aplicamos a regra de **1,5 × IQR** para sinalização de possíveis outliers. Eles são mantidos: valores extremos não representam necessariamente erros.

In [5]:
Q1 = df_train.quantile(0.25)
Q3 = df_train.quantile(0.75)
IQR = Q3 - Q1

outliers = (df_train < Q1 - 1.5 * IQR) | (df_train > Q3 + 1.5 * IQR)

display(outliers.sum())
display(df_train[outliers.any(axis=1)])

# Vamos manter essas linhas, pois não temos evidência de erro no registro.

TV           0
Radio        0
Newspaper    1
Sales        0
dtype: int64

,TV,Radio,Newspaper,Sales
101,296.4,36.3,100.9,23.8


## 2. Associação entre variáveis

Comparamos correlações de **Pearson** (lineares) e **Spearman** (por postos). Associação estatística não implica efeito causal nem determina, isoladamente, quais variáveis devem entrar no modelo.

In [6]:
# Correlação de Pearson entre todas as variáveis
correlacoes = df_train.corr(method='pearson')

display(correlacoes.round(3))

# Comparando Pearson e Spearman em relação às vendas
correlacoes_sales = pd.DataFrame({
      'Pearson': df_train.corr(method='pearson')['Sales'].drop('Sales')
    , 'Spearman': df_train.corr(method='spearman')['Sales'].drop('Sales')
})

display(correlacoes_sales.sort_values('Pearson', ascending=False).round(3))

,TV,Radio,Newspaper,Sales
TV,1.000,0.054,0.019,0.902
Radio,0.054,1.000,0.388,0.342
Newspaper,0.019,0.388,1.000,0.146
Sales,0.902,0.342,0.146,1.000


,Pearson,Spearman
TV,0.902,0.902
Radio,0.342,0.327
Newspaper,0.146,0.079


### Correlação parcial: rádio e vendas

Estimativa da associação entre `Radio` e `Sales` após controlar linearmente por `TV`. O ajuste estatístico não substitui um experimento causal.

In [7]:
correlacao_parcial = pg.partial_corr(
      data=df_train
    , x='Radio'
    , y='Sales'
    , covar='TV'
    , method='pearson'
)

display(correlacao_parcial)

,n,r,CI95,p_val
pearson,160,0.680655,"[0.59, 0.76]",5.646107e-23


### Correlação parcial: jornal e vendas

Estimativa da associação entre `Newspaper` e `Sales`, controlando por `TV` e `Radio`. Intervalos e p-valores são exploratórios, dado que as perguntas foram formuladas após observar os dados.

In [8]:
correlacao_parcial_jornal = pg.partial_corr(
      data=df_train
    , x='Newspaper'
    , y='Sales'
    , covar=['TV', 'Radio']
    , method='pearson'
)

display(correlacao_parcial_jornal)

,n,r,CI95,p_val
pearson,160,0.049469,"[-0.11, 0.2]",0.537063


## 3. Visualização exploratória

Distribuições e relações entre canais e vendas, usando **somente as 160 observações de treino**.

In [9]:
df_distribuicao = df_train.melt(
      var_name='Variável'
    , value_name='Valor'
)

fig = px.histogram(
      df_distribuicao
    , x='Valor'
    , facet_col='Variável'
    , nbins=15
    , title='Distribuição das variáveis'
    , labels={'Valor': 'Valor nas unidades do CSV'}
)

# Cada variável usa seus próprios intervalos no histograma.
for grafico in fig.data:
    grafico.bingroup = grafico.xaxis

fig.update_xaxes(matches=None)
fig.update_yaxes(title_text='Frequência')
fig.update_layout(width=1100, height=450)

fig.show()

### Investimentos por canal × vendas

A curva LOWESS resume tendências locais; não representa um modelo preditivo validado.

In [10]:
df_long = df_train.melt(
      id_vars='Sales'
    , value_vars=[  'TV'
                  , 'Radio'
                  , 'Newspaper'
            ]
    , var_name='Canal'
    , value_name='Investimento'
)

fig = px.scatter(
      df_long
    , x='Investimento'
    , y='Sales'
    , facet_col='Canal'
    , trendline='lowess'
    , trendline_options={'frac': 0.35}
    , title='Investimento e vendas'
    , labels={
              'Investimento': 'Investimento Publicitário'
            , 'Sales': 'Vendas'
        }
    , opacity=0.6
)

fig.update_xaxes(matches=None)

fig.update_layout(
      width=1100
    , height=450
)

fig.show()

### Relações entre canais

As cores representam o investimento em outro canal, permitindo observar padrões adicionais antes da comparação entre modelos.

In [11]:
fig = px.scatter(
    df_train
    , x='TV'
    , y='Sales'
    , color='Radio'
    , title='Vendas por Investimento em TV e Rádio'
    , labels={
                'TV': 'Investimento em TV'
              , 'Sales': 'Vendas'
              , 'Radio': 'Investimento em Rádio'
    }
    , opacity=0.6
    , trendline='lowess'
    , trendline_options={'frac': 0.35}
)

fig.update_layout(
        width=1100
        , height=450
)

fig.show()

In [12]:
fig = px.scatter(
    df_train
    , x='TV'
    , y='Sales'
    , color='Newspaper'
    , title='Vendas por Investimento em TV e Jornal'
    , labels={
                'TV': 'Investimento em TV'
              , 'Sales': 'Vendas'
              , 'Newspaper': 'Investimento em Jornal'
    }
    , opacity=0.6
    , trendline='lowess'
    , trendline_options={'frac': 0.35}
)

fig.update_layout(
        width=1100
        , height=450
)

fig.show()

In [13]:
fig = px.scatter(
    df_train
    , x='Radio'
    , y='Sales'
    , color='Newspaper'
    , title='Vendas por Investimento em Rádio e Jornal'
    , labels={
                'Radio': 'Investimento em Rádio'
              , 'Sales': 'Vendas'
              , 'Newspaper': 'Investimento em Jornal'
    }
    , opacity=0.6
    , trendline='lowess'
    , trendline_options={'frac': 0.35}
)

fig.update_layout(
        width=1100
        , height=450
)

fig.show()

Os gráficos sugerem uma associação mais forte de `TV` com vendas e possível contribuição adicional de `Radio`. A interação `TV × Radio` será testada **pela performance preditiva**, não inferida apenas da visualização.

## 4. Hipóteses de modelagem

| Pergunta | Comparação de modelos |
|---|---|
| TV melhora as previsões? | Só TV × previsão pela média |
| Rádio acrescenta informação? | TV + rádio × só TV |
| Jornal melhora a previsão? | Três canais × TV + rádio |
| Há ganho com interação? | TV × rádio × regressão aditiva |
| Maior flexibilidade ajuda? | Ridge polinomial e MLP × regressão linear |

São comparações **preditivas e exploratórias**, não testes de efeitos causais.

## 5. Modelos e validação

Usamos **RepeatedKFold (5 folds × 2 repetições)** no treino, com as mesmas divisões para todos os modelos. O critério de seleção é o **RMSE médio**; MAE e R² complementam a análise. O desvio do RMSE entre folds mede variabilidade observada, não intervalo de confiança.

In [14]:
validacao = RepeatedKFold(
      n_splits=5
    , n_repeats=2
    , random_state=42
)

# Usada apenas para escolher parâmetros dentro de cada treino da validação.
validacao_parametros = KFold(
      n_splits=4
    , shuffle=True
    , random_state=43
)

### Baselines e regressões lineares

Comparamos previsão pela média, regressões com diferentes conjuntos de canais e uma interação `TV × Radio`, construída apenas a partir das variáveis de entrada.

In [15]:
modelo_media = DummyRegressor(strategy='mean')
modelo_tv = LinearRegression()
modelo_tv_radio = LinearRegression()
modelo_linear = LinearRegression()
modelo_interacao = LinearRegression()

X_train_interacao = X_train.copy()
X_train_interacao['TV_Radio'] = X_train_interacao['TV'] * X_train_interacao['Radio']

### Ridge com termos polinomiais

Um `Pipeline` aplica expansão de grau 2, padronização e regressão Ridge. A busca por `alpha` ocorre **dentro do treino de cada divisão**, reduzindo vazamento de informação.

In [16]:
modelo_polinomial = Pipeline([
      ('termos', PolynomialFeatures(degree=2, include_bias=False))
    , ('escala', StandardScaler())
    , ('modelo', Ridge())
])

busca_polinomial = GridSearchCV(
      modelo_polinomial
    , param_grid={'modelo__alpha': [0.1, 1.0, 10.0, 100.0]}
    , scoring='neg_root_mean_squared_error'
    , cv=validacao_parametros
    , error_score='raise'
)

### Rede neural MLP

Testamos uma camada oculta de **5 ou 10 neurônios**, com `tanh`, otimizador `lbfgs` e diferentes valores de regularização. Entradas e alvo são padronizados **somente com os dados de treino** em cada divisão.

In [17]:
modelo_rn = Pipeline([
      ('escala', StandardScaler())
    , ('rede', MLPRegressor(
          activation='tanh'
        , solver='lbfgs'
        , max_iter=1500
        , random_state=42
    ))
])

modelo_rn = TransformedTargetRegressor(
      regressor=modelo_rn
    , transformer=StandardScaler()
)

parametros_rn = {
      'regressor__rede__hidden_layer_sizes': [(5,), (10,)]
    , 'regressor__rede__alpha': [0.1, 1.0]
}

busca_rn = GridSearchCV(
      modelo_rn
    , param_grid=parametros_rn
    , scoring='neg_root_mean_squared_error'
    , cv=validacao_parametros
    , error_score='raise'
)

### Validação cruzada aninhada

Para Ridge e MLP, o `GridSearchCV` é executado **dentro de cada fold externo** do `cross_validate`. A seleção de hiperparâmetros não acessa o fold de avaliação. MAE e RMSE são apresentados como erros positivos.

In [18]:
modelos = {
      'Média': modelo_media
    , 'Só TV': modelo_tv
    , 'TV + rádio': modelo_tv_radio
    , 'Três canais': modelo_linear
    , 'Interação TV × rádio': modelo_interacao
    , 'Polinômio': busca_polinomial
    , 'Rede neural': busca_rn
}

dados_modelos = {
      'Média': X_train
    , 'Só TV': X_train[['TV']]
    , 'TV + rádio': X_train[['TV', 'Radio']]
    , 'Três canais': X_train
    , 'Interação TV × rádio': X_train_interacao
    , 'Polinômio': X_train
    , 'Rede neural': X_train
}

metricas = {
      'RMSE': 'neg_root_mean_squared_error'
    , 'MAE': 'neg_mean_absolute_error'
    , 'R2': 'r2'
}

In [19]:
resultados_validacao = []
rmse_por_divisao = pd.DataFrame()

for nome, modelo in modelos.items():
    resultado = cross_validate(
          modelo
        , dados_modelos[nome]
        , Y_train
        , cv=validacao
        , scoring=metricas
        , return_train_score=True
        , error_score='raise'
    )

    rmse_por_divisao[nome] = -resultado['test_RMSE']

    resultados_validacao.append({
          'Modelo': nome
        , 'RMSE treino': -resultado['train_RMSE'].mean()
        , 'RMSE validação': -resultado['test_RMSE'].mean()
        , 'Desvio RMSE': resultado['test_RMSE'].std(ddof=1)
        , 'MAE validação': -resultado['test_MAE'].mean()
        , 'R² validação': resultado['test_R2'].mean()
    })

resultados_validacao = pd.DataFrame(resultados_validacao)
resultados_validacao = resultados_validacao.sort_values('RMSE validação')

display(resultados_validacao.round(3))

,Modelo,RMSE treino,RMSE validação,Desvio RMSE,MAE validação,R² validação
6,Rede neural,1.000,1.442,0.136,1.055,0.914
5,Polinômio,1.365,1.537,0.222,1.182,0.905
4,Interação TV × rádio,1.540,1.643,0.247,1.278,0.891
2,TV + rádio,1.630,1.679,0.278,1.279,0.885
3,Três canais,1.625,1.696,0.286,1.291,0.882
1,Só TV,2.231,2.271,0.162,1.827,0.791
0,Média,5.170,5.211,0.468,4.334,-0.067


A coluna `test_RMSE` retornada pelo scikit-learn representa a **validação de cada fold**, não as 40 observações do teste reservado. Diferenças entre erro de treino e validação ajudam a identificar possível sobreajuste.

In [20]:
fig = px.bar(
      resultados_validacao
    , x='RMSE validação'
    , y='Modelo'
    , orientation='h'
    , error_x='Desvio RMSE'
    , title='Comparação dos modelos — validação com 5 partes × 2 repetições'
    , labels={'RMSE validação': 'RMSE nas unidades de Sales; barras = ±1 desvio padrão'}
)

fig.update_yaxes(autorange='reversed')
fig.update_xaxes(rangemode='tozero')
fig.update_layout(width=1000, height=500)

fig.show()

### Ganho incremental das variáveis e dos modelos

A redução de RMSE é calculada nas **mesmas dez divisões**. Os folds repetidos não são independentes; a contagem de melhorias não deve ser interpretada como teste de significância.

In [21]:
ganhos = pd.DataFrame()

ganhos['TV sobre a média'] = rmse_por_divisao['Média'] - rmse_por_divisao['Só TV']
ganhos['Acrescentar rádio'] = rmse_por_divisao['Só TV'] - rmse_por_divisao['TV + rádio']
ganhos['Acrescentar jornal'] = rmse_por_divisao['TV + rádio'] - rmse_por_divisao['Três canais']
ganhos['Acrescentar interação'] = rmse_por_divisao['Três canais'] - rmse_por_divisao['Interação TV × rádio']
ganhos['Usar polinômio'] = rmse_por_divisao['Três canais'] - rmse_por_divisao['Polinômio']
ganhos['Usar rede neural'] = rmse_por_divisao['Três canais'] - rmse_por_divisao['Rede neural']

resumo_hipoteses = pd.DataFrame({
      'Ganho médio no RMSE': ganhos.mean()
    , 'Divisões com melhora (de 10)': (ganhos > 0).sum()
})

display(resumo_hipoteses.round(3))

,Ganho médio no RMSE,Divisões com melhora (de 10)
TV sobre a média,2.940,10
Acrescentar rádio,0.593,10
Acrescentar jornal,-0.017,2
Acrescentar interação,0.054,7
Usar polinômio,0.159,9
Usar rede neural,0.254,7


### Ajuste do modelo selecionado

O modelo com menor RMSE médio é ajustado novamente às **160 observações de treino**. A busca de parâmetros usa apenas esse conjunto. A escolha entre famílias de modelos ainda pode favorecer a melhor pontuação da validação.

In [22]:
nome_modelo = resultados_validacao.iloc[0]['Modelo']
modelo = modelos[nome_modelo]

modelo.fit(dados_modelos[nome_modelo], Y_train)

print(f'Modelo escolhido: {nome_modelo}')

if isinstance(modelo, GridSearchCV):
    print(f'Parâmetros escolhidos: {modelo.best_params_}')

Modelo escolhido: Rede neural
Parâmetros escolhidos: {'regressor__rede__alpha': 0.1, 'regressor__rede__hidden_layer_sizes': (10,)}


## 6. Avaliação no teste reservado

Avaliamos o modelo escolhido nas **40 observações preservadas** e comparamos os resultados com os baselines de média e regressão linear com três canais. Esses resultados não serão usados para selecionar outro modelo.

In [23]:
X_test_modelo = X_test.copy()

if nome_modelo == 'Interação TV × rádio':
    X_test_modelo['TV_Radio'] = X_test_modelo['TV'] * X_test_modelo['Radio']

X_test_modelo = X_test_modelo[dados_modelos[nome_modelo].columns]
Y_pred = modelo.predict(X_test_modelo)

mae = mean_absolute_error(Y_test, Y_pred)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
r2 = r2_score(Y_test, Y_pred)

print(f'Modelo: {nome_modelo}')
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²: {r2:.4f}')

Modelo: Rede neural
MAE: 0.9608
RMSE: 1.2578
R²: 0.9488


In [24]:
modelo_media.fit(X_train, Y_train)
modelo_linear.fit(X_train, Y_train)

previsoes_teste = {
      'Média': modelo_media.predict(X_test)
    , 'Três canais': modelo_linear.predict(X_test)
}
previsoes_teste[nome_modelo] = Y_pred

resultados_teste = []

for nome, previsao in previsoes_teste.items():
    resultados_teste.append({
          'Modelo': nome
        , 'MAE': mean_absolute_error(Y_test, previsao)
        , 'RMSE': np.sqrt(mean_squared_error(Y_test, previsao))
        , 'R²': r2_score(Y_test, previsao)
    })

display(pd.DataFrame(resultados_teste).round(3))

,Modelo,MAE,RMSE,R²
0,Média,4.932,5.648,-0.032
1,Três canais,1.275,1.705,0.906
2,Rede neural,0.961,1.258,0.949


### Valores previstos × observados

A diagonal indica previsão perfeita; ambos os eixos usam a mesma escala.

In [25]:
resultados = pd.DataFrame({
      'Y_test': Y_test
    , 'Y_pred': Y_pred
})

fig = px.scatter(
      resultados
    , x='Y_test'
    , y='Y_pred'
    , title=f'Valores reais × previstos — {nome_modelo} (40 linhas de teste)'
    , labels={'Y_test': 'Vendas reais', 'Y_pred': 'Vendas previstas'}
    , opacity=0.7
)

limite_min = resultados[['Y_test', 'Y_pred']].min().min() - 1
limite_max = resultados[['Y_test', 'Y_pred']].max().max() + 1

fig.update_xaxes(range=[limite_min, limite_max])
fig.update_yaxes(range=[limite_min, limite_max], scaleanchor='x', scaleratio=1)

fig.add_shape(
      type='line'
    , x0=limite_min, y0=limite_min
    , x1=limite_max, y1=limite_max
    , line=dict(color='orange', dash='dash')
)

fig.update_layout(width=750, height=750)

fig.show()

### Resíduos e maiores erros

Resíduo = valor real − valor previsto. Padrões sistemáticos podem indicar limitações do ajuste; examinamos os maiores erros sem descartá-los.

In [26]:
resultados['Residuo'] = resultados['Y_test'] - resultados['Y_pred']

fig = px.scatter(
      resultados
    , x='Y_pred'
    , y='Residuo'
    , title=f'Análise de resíduos — {nome_modelo} (teste)'
    , labels={'Y_pred': 'Vendas previstas', 'Residuo': 'Resíduo nas unidades de Sales'}
    , opacity=0.7
)

fig.add_hline(y=0, line_dash='dash', line_color='orange')
fig.update_layout(width=900, height=450)

fig.show()

In [27]:
resultados['Erro_absoluto'] = resultados['Residuo'].abs()

# O índice mantém a ligação com os investimentos da mesma linha.
maiores_erros = X_test.join(resultados)

display(maiores_erros.sort_values('Erro_absoluto', ascending=False).head())
print(f'Resíduo médio: {resultados["Residuo"].mean():.3f}')

,TV,Radio,Newspaper,Y_test,Y_pred,Residuo,Erro_absoluto
150,280.7,13.9,37.0,16.1,19.913533,-3.813533,3.813533
66,31.5,24.6,2.2,11.0,8.300509,2.699491,2.699491
16,67.8,36.6,114.0,12.5,14.745137,-2.245137,2.245137
152,197.6,23.3,14.2,16.6,18.718999,-2.118999,2.118999
60,53.5,2.0,21.4,8.1,9.920755,-1.820755,1.820755


Resíduo médio: -0.375


## 7. Exemplo de previsão

Estimativa ilustrativa para investimentos dentro das faixas observadas no treino. Não é uma recomendação de alocação de orçamento.

In [28]:
novo_investimento = pd.DataFrame({
      'TV': [150.0]
    , 'Radio': [25.0]
    , 'Newspaper': [30.0]
})

if nome_modelo == 'Interação TV × rádio':
    novo_investimento['TV_Radio'] = novo_investimento['TV'] * novo_investimento['Radio']

novo_investimento = novo_investimento[dados_modelos[nome_modelo].columns]
nova_previsao = modelo.predict(novo_investimento)

print(f'Vendas previstas: {nova_previsao[0]:.2f} nas unidades do CSV')

Vendas previstas: 15.67 nas unidades do CSV


## 8. Resultados e limitações

**Modelo selecionado nesta execução:** MLP com **10 neurônios em uma camada oculta** e `alpha = 0,1` (escolha do `GridSearchCV`). A rede obteve **RMSE médio de 1,442** na validação cruzada, contra **1,537** do Ridge polinomial e **1,696** da regressão linear com três canais.

**Teste reservado (40 observações):** **MAE = 0,9608**, **RMSE = 1,2578** e **R² = 0,9488**. A regressão linear com três canais teve **RMSE = 1,705** nesse mesmo teste. A diferença entre o RMSE de treino da MLP (**1,000**) e o de validação (**1,442**) sugere atenção ao possível sobreajuste.

**Análise dos canais:** adicionar rádio à TV melhorou o RMSE nas dez divisões avaliadas; adicionar jornal à regressão aditiva melhorou apenas duas, sem ganho médio nesta amostra. Isso **não prova** que jornal seja irrelevante em outros contextos.

**Limitações:** amostra pequena (200 linhas), teste com apenas 40 casos, origem e unidades da base não documentadas, possível dependência entre observações e ausência de variáveis como preço, sazonalidade e concorrência. As relações estimadas são **preditivas, não causais**; a utilidade em produção exigiria dados novos, validação externa e monitoramento.

**Referências:** [Pipeline](https://scikit-learn.org/stable/modules/compose.html) · [Validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html) · [MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).